In [43]:
import pandas as pd

df = pd.read_csv("complaints.csv")


C:\Users\Drashti Shah\AppData\Local\Temp\ipykernel_20544\56437378.py:3: DtypeWarning: Columns (0: Consumer complaint narrative) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("complaints.csv")


In [44]:
df['Product'].value_counts()

Product
Credit reporting or other personal consumer reports                             10308722
Credit reporting, credit repair services, or other personal consumer reports     2163774
Debt collection                                                                  1111106
Mortgage                                                                          450346
Checking or savings account                                                       369971
Credit card                                                                       316892
Credit card or prepaid card                                                       206356
Money transfer, virtual currency, or money service                                180430
Credit reporting                                                                  140426
Student loan                                                                      127976
Vehicle loan or lease                                                              95981
Bank account 

In [45]:
df.shape

(15694521, 16)

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15694521 entries, 0 to 15694520
Data columns (total 16 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Date received                 str  
 1   Product                       str  
 2   Sub-product                   str  
 3   Issue                         str  
 4   Sub-issue                     str  
 5   Consumer complaint narrative  str  
 6   Company public response       str  
 7   Company                       str  
 8   State                         str  
 9   ZIP code                      str  
 10  Tags                          str  
 11  Submitted via                 str  
 12  Date sent to company          str  
 13  Company response to consumer  str  
 14  Timely response?              str  
 15  Complaint ID                  int64
dtypes: int64(1), str(15)
memory usage: 9.5 GB


In [47]:
df['Issue'].sample(5)

1469096     Incorrect information on your report
13490729    Incorrect information on your report
14901131             Improper use of your report
4116262     Incorrect information on your report
8348499     Incorrect information on your report
Name: Issue, dtype: str

In [48]:
df.isnull().sum()

Date received                          0
Product                                0
Sub-product                       235275
Issue                                  6
Sub-issue                         911946
Consumer complaint narrative    11901278
Company public response          7223685
Company                                0
State                              60941
ZIP code                             645
Tags                            14934339
Submitted via                          0
Date sent to company                   0
Company response to consumer          23
Timely response?                       0
Complaint ID                           0
dtype: int64

In [49]:
df.shape[0]-df['Consumer complaint narrative'].isnull().sum()

np.int64(3793243)

In [50]:
df['Consumer complaint narrative']

0                                                         NaN
1           I cancelled XXXX services. They claimed I had ...
2           VSG Contacted my step-daughter and told her I ...
3           LAst week, I was contacted by the debt collect...
4                                                         NaN
                                  ...                        
15694516    I am a natural person, I am a living human bei...
15694517                                                  NaN
15694518                                                  NaN
15694519                                                  NaN
15694520                                                  NaN
Name: Consumer complaint narrative, Length: 15694521, dtype: str

In [51]:
## CFPB only publishes narratives when the consumer explicitly consents to public disclosure. Most consumers don't opt in.
## That's why so much missing values in Consumer Complaint Narrative

In [52]:
df = df[['Product','Date received','Consumer complaint narrative','Issue','Timely response?']]

In [53]:
df = df.dropna(subset=['Consumer complaint narrative'])

In [54]:
df.isnull().sum()

Product                         0
Date received                   0
Consumer complaint narrative    0
Issue                           0
Timely response?                0
dtype: int64

In [55]:
df['Timely response?'].value_counts()

Timely response?
Yes    3752508
No       40735
Name: count, dtype: int64

In [56]:
df.shape

(3793243, 5)

In [57]:
df['Product'].value_counts()

Product
Credit reporting or other personal consumer reports                             1670071
Credit reporting, credit repair services, or other personal consumer reports     807501
Debt collection                                                                  426986
Checking or savings account                                                      179051
Mortgage                                                                         143407
Credit card                                                                      119445
Money transfer, virtual currency, or money service                               116815
Credit card or prepaid card                                                      108683
Student loan                                                                      61147
Vehicle loan or lease                                                             50808
Credit reporting                                                                  31587
Payday loan, title loan,

In [58]:
product_mapping = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Debt collection': 'Debt Collection',
    'Debt or credit management': 'Debt Collection',
    'Mortgage': 'Loans',
    'Student loan': 'Loans',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan': 'Loans',
    'Credit card': 'Cards & Transactions',
    'Credit card or prepaid card': 'Cards & Transactions',
    'Prepaid card': 'Cards & Transactions',
    'Checking or savings account': 'Banking & Accounts',
    'Bank account or service': 'Banking & Accounts',
    'Money transfer, virtual currency, or money service': 'Money Transfer & Payments',
    'Money transfers': 'Money Transfer & Payments',
}

df['Category'] = df['Product'].map(product_mapping)

df = df.dropna(subset = ['Category'] )

In [59]:
df['Category'].value_counts()

Category
Credit Reporting             2509159
Debt Collection               432038
Loans                         300300
Cards & Transactions          239192
Banking & Accounts            193934
Money Transfer & Payments     118312
Name: count, dtype: int64

In [60]:
df['Category'].isnull().sum()

np.int64(0)

### Class imbalance handled via class_weight='balanced' in classifier rather than resampling, to preserve natural complaint distribution.

In [61]:
df[df['Category'] == 'Money Transfer & Payments']['Consumer complaint narrative'].sample(1).values

<ArrowStringArray>
['my account was hacked and social security card, identification card, birth certificate, and debit cards along with phone was stolen and used to access my accounts and make transactions I didn't know about. I called PayPal and informed them of the problem but was refused access to my Account because my information had been changed and I could no longer verify the account. my money was stolen from XX/XX/2023 until XX/XX/2023 when I was finally able to regain access to my account with the help of a PayPal representative. I filed disputes against the unauthorized charges and they were automatically denied by PayPal without an Investigation I was told by several representatives. PayPal then took. oney from my account After informing me that XXXX cases would be closed I my favor and refunded to my account instead of crediting my account with the amounts PayPal debited the amounts fro. my account bringing my account to a negative balance. again they brought my account int

In [7]:
import re
import nltk 
from nltk.corpus import stopwords,wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer

# Download everything needed for tokenization, tagging, and lemmatization
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')                          # Digital lexicon backup for wordnet
nltk.download('averaged_perceptron_tagger')       # Legacy tagger weights
nltk.download('averaged_perceptron_tagger_eng')   # Updated tagger weights

[nltk_data] Downloading package punkt to C:\Users\Drashti
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Drashti
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Drashti
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Drashti
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Drashti
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Drashti Shah\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is 

True

In [8]:
def get_wordnet_pos(tag):
    """Maps Penn Treebank POS tags to WordNet POS tags"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN # Default fallback to noun

In [9]:
import contractions as ct

stop_words = set(stopwords.words('english'))

negation_words = {'no','not','never','neither','nor','nothing','nowhere','nobody','cannot','without','against'}

stop_words = stop_words - negation_words
lemmatizer = WordNetLemmatizer()

def clean_preprocessed_text(text):

    text = ct.fix(text)

    # text = re.sub(r'\bnt\b', '', text)  
    
    text = re.sub(r'(\d+|X+)/(\d+|X+)/(\d+|X+|year>)','',text)

    text = re.sub(r'\{\$\d+\.\d+\}','',text)

    text = re.sub(r'X{2,}','',text)

    text = text.lower()
    
    text = re.sub(r'[\n\t]+', ' ', text)

    text = re.sub(r'[^\w\s]','',text) 

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    pos_tags = pos_tag(tokens)

    cleaned_tokens = []

    for word,tag in pos_tags:

        wn_tag = get_wordnet_pos(tag)

        lemma = lemmatizer.lemmatize(word,pos = wn_tag)

        if lemma not in stop_words and len(lemma) > 1 and not lemma.isnumeric():
            cleaned_tokens.append(lemma)

        
    return " ".join(cleaned_tokens)

In [65]:
sample = df['Consumer complaint narrative'].iloc[100000]
print(clean_preprocessed_text(sample))

request regard inaccurate unknown account credit report multiple error show report please verify account percent true accurate correct not untimely unverifiable incomplete inaccurate please correct update account fix negativity comply fcra section require reporting accurate depted balance owe depted balance owe


In [66]:
from tqdm import tqdm
tqdm.pandas()


df['cleaned_narrative'] = df['Consumer complaint narrative'].progress_apply(clean_preprocessed_text)

100%|██████████| 3792935/3792935 [4:54:35<00:00, 214.59it/s]  


In [68]:
df['cleaned_narrative'].iloc[100000]

'request regard inaccurate unknown account credit report multiple error show report please verify account percent true accurate correct not untimely unverifiable incomplete inaccurate please correct update account fix negativity comply fcra section require reporting accurate depted balance owe depted balance owe'

In [69]:
df.to_csv('cleaned_complaints.csv', index=False)

## EDA

In [2]:
## I closed the IDE here so load again 
import pandas as pd
df = pd.read_csv('cleaned_complaints.csv')

In [2]:
df['cleaned_narrative'].isna().sum()

np.int64(0)

In [3]:
df['Category'].value_counts()

Category
Credit Reporting             2392911
Debt Collection               414682
Loans                         295386
Cards & Transactions          234054
Banking & Accounts            188907
Money Transfer & Payments     115693
Name: count, dtype: int64

In [4]:
df['Category'].value_counts(normalize = True)*100

Category
Credit Reporting             65.709834
Debt Collection              11.387254
Loans                         8.111361
Cards & Transactions          6.427172
Banking & Accounts            5.187426
Money Transfer & Payments     3.176954
Name: proportion, dtype: float64

#### 66% Credit Reporting is a heavy imbalance but  even 3% of 3.79 million is 113,000 rows which is substantial training data for any category.
#### This is why class_weight='balanced' is the right call rather than resampling. You have enough data in every class, just unequally distributed.


#### With 66% Credit Reporting, a model that predicts Credit Reporting for everything achieves 66% accuracy while being completely useless.
#### So using weighted F1 score as your primary metric is the right call because it accounts for class imbalance and gives a honest picture of model performance across all categories.

In [5]:
df['cleaned_narrative'].iloc[0]

'cancel service claim contract could not cancel ask copy sign contract send contract without signature claim verbal contract since phone system state call record ask send copy recording enter verbal contract never receive either move already instal never sign contract continue pay realize not use could save money cancel call cancel tell first write contract could not produce verbal contract cannothave nt produce send virtuoso collection send letter virtuoso dispute charge request copy write verbal contract not not provide continue call multiple time per day even though letter say stop call dispute addition call ask correspond write request copy allege contract follow rule virtuoso also resent letter virtuoso virtually produce valid contract show enter agreement pay amount due ask virtuoso contract month verbally write neither produce anything'

In [3]:
df['cleaned_narrative'].apply(type).value_counts()

cleaned_narrative
<class 'str'>    3641633
Name: count, dtype: int64

In [4]:
df['token_count'] = df['cleaned_narrative'].apply(lambda x : len(x.split()))

In [8]:
token_count_per_cat_avg = df.groupby('Category')['token_count'].mean()

In [9]:
token_count_per_cat_avg

Category
Banking & Accounts           104.258376
Cards & Transactions         101.227785
Credit Reporting              77.901043
Debt Collection               80.742501
Loans                        121.500799
Money Transfer & Payments     86.943454
Name: token_count, dtype: float64

In [10]:
token_count_per_cat_min = df.groupby('Category')['token_count'].min()
token_count_per_cat_min

Category
Banking & Accounts           10
Cards & Transactions         10
Credit Reporting             10
Debt Collection              10
Loans                        10
Money Transfer & Payments    10
Name: token_count, dtype: int64

In [11]:
from collections import Counter 

top_words_per_category = {}

for category, group in df.groupby('Category'):

    text = ' '.join(group['cleaned_narrative'].dropna())

    word_counts = Counter(text.split())

    top_words_per_category[category] = word_counts.most_common(20)


for category, words in top_words_per_category.items():
    print(f"\n{category}")
    print(words)




Banking & Accounts
[('account', 693050), ('not', 531375), ('bank', 385231), ('check', 225608), ('call', 210480), ('money', 173314), ('tell', 167908), ('fund', 162703), ('no', 157703), ('would', 154869), ('receive', 144649), ('transaction', 142816), ('deposit', 139661), ('day', 135976), ('say', 130332), ('get', 126741), ('card', 126564), ('charge', 124049), ('time', 123970), ('make', 119791)]

Cards & Transactions
[('not', 641792), ('card', 541314), ('credit', 505480), ('account', 466102), ('payment', 277304), ('charge', 268569), ('call', 249848), ('bank', 196822), ('make', 192024), ('receive', 190881), ('pay', 184588), ('would', 183813), ('no', 174194), ('tell', 164375), ('time', 162830), ('one', 143582), ('say', 140250), ('dispute', 140187), ('get', 134385), ('balance', 132888)]

Credit Reporting
[('report', 6172135), ('credit', 6150821), ('account', 5505401), ('not', 3988021), ('information', 3775308), ('consumer', 3390737), ('reporting', 2537014), ('payment', 1467516), ('dispute', 

In [ ]:
word_count = Counter()

for category,word_tuples in top_words_per_category.items():

    word = [word for word,count in word_tuples]

    word_count.update(word)


filtered_words = {
    word:count
    for word,count in word_count.items() if count>=3
}

print(filtered_words)

{'account': 6, 'not': 6, 'bank': 3, 'call': 5, 'tell': 3, 'no': 5, 'would': 3, 'receive': 4, 'say': 3, 'get': 3, 'time': 3, 'make': 3, 'credit': 4, 'payment': 4, 'pay': 3, 'dispute': 4, 'consumer': 3, 'request': 3, 'send': 3}


In [6]:
from collections import Counter
from nltk.util import bigrams

top_bigram_per_cat = {}

for category, group in df.groupby('Category'):

    bigram_counter = Counter()

    for text in group['cleaned_narrative'].dropna():

        tokens = text.split()
        bigram_counter.update(bigrams(tokens))

    top_bigram_per_cat[category] = bigram_counter.most_common(15)


print(top_bigram_per_cat)

{'Banking & Accounts': [(('well', 'fargo'), 66022), (('check', 'account'), 48455), (('bank', 'america'), 45779), (('debit', 'card'), 40207), (('close', 'account'), 40006), (('customer', 'service'), 34641), (('could', 'not'), 33311), (('bank', 'account'), 30748), (('capital', 'one'), 26840), (('account', 'not'), 25854), (('checking', 'account'), 25502), (('saving', 'account'), 24697), (('account', 'close'), 24552), (('open', 'account'), 23998), (('overdraft', 'fee'), 23992)], 'Cards & Transactions': [(('credit', 'card'), 207421), (('capital', 'one'), 70686), (('customer', 'service'), 48430), (('american', 'express'), 39803), (('credit', 'report'), 39210), (('bank', 'america'), 35184), (('make', 'payment'), 34862), (('could', 'not'), 34841), (('late', 'fee'), 34318), (('close', 'account'), 31501), (('late', 'payment'), 29262), (('not', 'receive'), 29050), (('would', 'not'), 26826), (('card', 'account'), 25010), (('well', 'fargo'), 22871)], 'Credit Reporting': [(('credit', 'report'), 2529

In [7]:
top_words_per_category = {}

for category, group in df.groupby('Category'):

    text = ' '.join(group['cleaned_narrative'].dropna())

    word_counts = Counter(text.split())

    top_words_per_category[category] = word_counts.most_common(50)


category_word_sets = {
    category: {word for word, count in words}
    for category, words in top_words_per_category.items()
}


for category, word_set in category_word_sets.items():

    # Union of all other categories' word sets
    other_words = set().union(
        *(s for cat, s in category_word_sets.items() if cat != category)
    )

    # Words unique to this category
    exclusive = word_set - other_words

    print(f"\n{category}:")
    print(sorted(exclusive))



Banking & Accounts:
['chase', 'deposit', 'well']

Cards & Transactions:
['purchase', 'statement']

Credit Reporting:
['bureau', 'delete', 'fcra', 'fraudulent', 'identity', 'inaccurate', 'inquiry', 'item', 'list', 'section', 'theft', 'usc', 'violation', 'without', 'yousc']

Debt Collection:
['collect', 'collection', 'original', 'owe', 'validation']

Loans:
['document', 'home', 'loan', 'mortgage', 'need', 'since', 'year']

Money Transfer & Payments:
['address', 'app', 'cash', 'fail', 'financial', 'leave', 'paypal', 'platform', 'process', 'resolution', 'significant', 'unfair', 'vulnerable', 'zelle']


#### Experimentated with different models but apparantely the notebook data got lost so only the final is written now

In [9]:
df_sample = (
    df.groupby('Category', group_keys=False)
      .sample(n=33000, random_state=42)
      .reset_index(drop=True)
)
print(df_sample.shape)
print(df_sample['Category'].value_counts())

(198000, 9)
Category
Banking & Accounts           33000
Cards & Transactions         33000
Credit Reporting             33000
Debt Collection              33000
Loans                        33000
Money Transfer & Payments    33000
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [11]:
X = df_sample['cleaned_narrative']
Y = df_sample['Category']

x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [12]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=100000,
    min_df=5,
    sublinear_tf=True
)
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

In [13]:
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=42
)
lr.fit(x_train_vec, y_train)

c:\Users\Drashti Shah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [14]:
y_pred = lr.predict(x_test_vec)
print(classification_report(y_test, y_pred))

                           precision    recall  f1-score   support

       Banking & Accounts       0.80      0.84      0.82      6600
     Cards & Transactions       0.84      0.83      0.84      6600
         Credit Reporting       0.85      0.88      0.86      6600
          Debt Collection       0.86      0.84      0.85      6600
                    Loans       0.89      0.89      0.89      6600
Money Transfer & Payments       0.89      0.86      0.87      6600

                 accuracy                           0.86     39600
                macro avg       0.86      0.86      0.86     39600
             weighted avg       0.86      0.86      0.86     39600



In [15]:
import joblib

joblib.dump(lr, 'lr_model_final.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer_100k.pkl')

['tfidf_vectorizer_100k.pkl']

In [17]:
df['Consumer complaint narrative'].iloc[0]

'I cancelled XXXX services. They claimed I had a contract with them and couldn\'t cancel until XXXX. When I asked for a copy of my signed contract, they sent me an XXXX contract without my signature. XXXX them claimed that it was a verbal contract. Since XXXX phone system states that calls are recorded, I asked them to send me a copy of the recording were I entered into a verbal contract. I never received this either. I moved into my in XXXX and XXXX was already installed. We never signed a contract, we continued to pay until XXXX when we realized we didn\'t use it and could save money by cancelling it. When I called to cancel, this is when they told me first that I had a written contract and when they couldn\'t produce that, that I had a verbal contract ( which again, they can\'t/have n\'t produced ). They sent to Virtuoso Collections. On XX/XX/XXXX, I sent a letter to Virtuoso disputing the charge and requesting a copy of written or verbal contract from XXXX. Not only did they not pr

In [1]:
import pandas as pd

df = pd.read_csv('cleaned_complaints.csv', encoding='utf-8', engine='python')

In [11]:
df.head()

,Unnamed: 0,Product,Date received,Consumer complaint narrative,Issue,Timely response?,Category,cleaned_narrative,token_count
0,0,Debt collection,2023-04-28T13:09:35.000Z,I cancelled XXXX services. They claimed I had ...,Attempts to collect debt not owed,No,Debt Collection,cancel service claim contract could not cancel...,127
1,1,Debt collection,2023-06-16T18:07:02.000Z,VSG Contacted my step-daughter and told her I ...,Threatened to contact someone or share informa...,No,Debt Collection,vsg contact stepdaughter tell bill get ready g...,26
2,2,Debt collection,2023-07-25T21:03:22.000Z,"LAst week, I was contacted by the debt collect...",Communication tactics,No,Debt Collection,last week contact debt collector virtuoso sour...,107
3,3,Credit reporting or other personal consumer re...,2024-11-12T17:40:48.000Z,I received yet another letter on XX/XX/year> c...,Unable to get your credit report or credit score,Yes,Credit Reporting,receive yet another letter case rthey refuse m...,44
4,4,Credit reporting or other personal consumer re...,2024-12-09T18:23:28.000Z,I have spoken to Lexus nexus Multipke times. X...,Problem with a company's investigation into an...,Yes,Credit Reporting,speak lexus nexus multipke time assure would m...,20


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3641633 entries, 0 to 3641632
Data columns (total 9 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Unnamed: 0                    int64
 1   Product                       str  
 2   Date received                 str  
 3   Consumer complaint narrative  str  
 4   Issue                         str  
 5   Timely response?              str  
 6   Category                      str  
 7   cleaned_narrative             str  
 8   token_count                   int64
dtypes: int64(2), str(7)
memory usage: 6.3 GB


In [13]:
df['Consumer complaint narrative'][0]

'I cancelled XXXX services. They claimed I had a contract with them and couldn\'t cancel until XXXX. When I asked for a copy of my signed contract, they sent me an XXXX contract without my signature. XXXX them claimed that it was a verbal contract. Since XXXX phone system states that calls are recorded, I asked them to send me a copy of the recording were I entered into a verbal contract. I never received this either. I moved into my in XXXX and XXXX was already installed. We never signed a contract, we continued to pay until XXXX when we realized we didn\'t use it and could save money by cancelling it. When I called to cancel, this is when they told me first that I had a written contract and when they couldn\'t produce that, that I had a verbal contract ( which again, they can\'t/have n\'t produced ). They sent to Virtuoso Collections. On XX/XX/XXXX, I sent a letter to Virtuoso disputing the charge and requesting a copy of written or verbal contract from XXXX. Not only did they not pr

In [ ]:
severity_keywords = {
    "Critical": [
        "identity theft",
        "identity theft report",
        "eviction",
        "evicted",
        "eviction notice",
        "nsf",
        "account closed without",
        "facing eviction",
        "late fees accumulating",
        "account hacked",
        "hacked",
        "account takeover",
        "takeover",
        "unauthorized access",
        "unauthorized withdrawal",
        "unauthorized transaction",
        "frozen account",
        "frozen",
        "data breach",
        "breach",
        "compromised",
        "legal action",
        "foreclosure",
        "drained",
        "emptied",
        "stolen",
        "medical emergency",
        "emergency",
        "cannot pay rent",
        "unable to buy food",
        "utility shutoff",
        "fraudulent",
        "cannot access",
        "rent due",
        "no other way to pay",
        "due tomorrow",
        "funds locked",
        "not access",
        "no way pay",
        "fund lock"
    ],

    "High": [
        "fraud",
        "theft",
        "victim",
        "unauthorized charge",
        "credit report error",
        "credit damage",
        "score dropped",
        "late fee",
        "payment dispute",
        "account restriction",
        "restriction",
        "financial loss",
        "loss",
        "refused refund",
        "refund",
        "unresolved dispute",
        "harassment",
        "multiple calls",
        "supervisor",
        "escalate",
        "repeatedly ignored",
        "ignored",
        "repossession",
        "overdraft",
        "charged-off",
        "credit denied",
        "loan denied",
        "mortgage denied",
        "wage garnishment",
        "being targeted",
        "unknown creditor",
        "do not know who",
        "collection scam",
        "suspicious email",
        "not authorized",
        "without my consent",
        "not approve",
        "blocked and removed",
        "did not approve",
        "reported as delinquent",
        "credit score dropped",
        "applied to wrong account",
        "on time payment",
        "misapplied",
        "report delinquent",
        "apply wrong account",
        "misapply",
        "money disappeared",
        "money lost",
        "money missing",
        "never arrived",
        "never received",
        "lost money",

    ],

    "Medium": [
        "incorrect information",
        "billing confusion",
        "confusion",
        "customer service",
        "long wait",
        "delayed response",
        "statement error",
        "technical issue",
        "technical",
        "service interruption",
        "interruption",
        "communication problem",
        "communication",
        "verification issue",
        "verification",
        "validate",
        "validation",
        "do not owe",
        "billing error",
        "billing",
        "wrong charge",
        "app not working",
        "login issue",
        "login",
        "incorrect charge",
        "delayed",
        "not received",
        "website",
        "processing",
        "remove from credit report",
        "remove from my report",
        "remove account",
        "second request",
        "second time",
        "fcra",
        "violation",
        "without my knowledge",
        "without consent",
        "did not authorize",
        "unauthorized entries",
        "not give written permission",
        "already paid",
        "have receipt",
        "paid in full",
        "collections notice",
        "paid two uears ago",
        "proof of payment",
        "already pay",
        "collection notice",
        "receipt",
        "pay full",
        "proof payment"
    ],

    "Low": [
        "address update",
        "document request",
        "status inquiry",
        "inquiry",
        "general inquiry",
        "information request",
        "account maintenance",
        "maintenance",
        "profile update",
        "profile",
        "password reset",
        "reset",
        "application status",
        "account verification",
        "please confirm",
        "requesting statement",
        "account setup",
        "setup",
        "how do i",
        "would like to know",
        "clarification",
        "assistance",
        "question",
        "hard inquiry",
        "hard inquiries"
    ]
}

severity_weights = {
    "Critical": 4,
    "High": 3,
    "Medium": 2,
    "Low": 1
}


def urgency_level(text, return_details=False):

    text = str(text).lower()

    scores = {
        "Critical": 0,
        "High": 0,
        "Medium": 0,
        "Low": 0
    }

    matched_keywords = {
        "Critical": [],
        "High": [],
        "Medium": [],
        "Low": []
    }

    for level, keywords in severity_keywords.items():
        for keyword in keywords:
            if keyword in text:
                scores[level] += severity_weights[level]
                matched_keywords[level].append(keyword)

    # No keyword matched
    if sum(scores.values()) == 0:
        severity = "Low"

    # Critical events should dominate
    elif scores["Critical"] >= 8:
        severity = "Critical"

    # Strong High signal
    elif scores["High"] >= 6:
        severity = "High"

    # Medium signal
    elif scores["Medium"] >= 4:
        severity = "Medium"

    # Fallback to highest score
    else:
        priority = ["Critical", "High", "Medium", "Low"]

        max_score = max(scores.values())

        for level in priority:
            if scores[level] == max_score:
                severity = level
                break

    if return_details:
        return {
            "severity": severity,
            "scores": scores,
            "matched_keywords": matched_keywords
        }

    return severity

In [20]:
sample = df['Consumer complaint narrative'].iloc[110]
x = urgency_level(sample)
print(x)

High


In [55]:
print(sample)

Bank of America is reporting my credit card account ending in XXXX as charged-off on my credit reports. 

I disputed this account directly with Bank of America and requested verification of the accuracy and legal basis for the charge-off. Bank of America indicated that it provided a response, but no written response was ever received by mail, email, or secure message. 

Bank of America has failed to provide proof that I was notified within 30 days of furnishing negative information, as required under the Fair Credit Reporting Act. The bank has also failed to clarify whether the debt was cancelled or remains legally enforceable. 

Because no response or verification was delivered, the charge-off reporting remains inaccurate and unresolved, yet continues to appear on my credit reports.


In [56]:
text = sample.lower()

for level, keywords in severity_keywords.items():
    matched = [kw for kw in keywords if kw in text]
    print(f"{level}: {matched}")

Critical: []
High: ['charged-off']
Medium: ['verification']
Low: []


In [3]:
sample_check = df.sample(20, random_state=42)[['Consumer complaint narrative', 'Category']]
sample_check['urgency'] = sample_check['Consumer complaint narrative'].apply(urgency_level)

In [4]:
for i in range(0,20):
    print(f"{i}: {sample_check['Consumer complaint narrative'].iloc[i]}")

0: I have a small business account ( XXXX XXXX XXXX ). On XX/XX/XXXX, were made 4 unauthorized ACHs from my account to a company called XXXX XXXX, in value of {$490.00} each. I have no idea what is this company. These transfers were frauds. So, I called Bank of America and proceeded in the manner they instructed me. After that, I only received one email saying they will investigate my case and will return in 60 business days. I think this time is unreasonable and the email is vague. I can't wait for more than 2 months to have my money back that was stolen from my account. I have bills to pay and I understand that I trust my money in the Bank, I pay fees to have security, so my money can't simply vanish from my account, without explanation for so long time. I started making many calls trying to understand if the Bank was really investigating my case and when I would receive my money back. Believe it or not, I called almost 10 different numbers that the Bank provided me, trying to get tr

In [5]:
for i in range(0,20):
    print(f"{i}: {sample_check['urgency'].iloc[i]}")

0: Critical
1: Medium
2: Medium
3: Medium
4: Critical
5: High
6: High
7: Low
8: Medium
9: High
10: Critical
11: Critical
12: Critical
13: Critical
14: Medium
15: Medium
16: Low
17: Medium
18: High
19: Critical


In [6]:
from tqdm import tqdm
tqdm.pandas()

df['Urgency'] = df['Consumer complaint narrative'].progress_apply(urgency_level)

100%|██████████| 3641633/3641633 [07:34<00:00, 8007.35it/s] 


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3641633 entries, 0 to 3641632
Data columns (total 10 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Unnamed: 0                    int64
 1   Product                       str  
 2   Date received                 str  
 3   Consumer complaint narrative  str  
 4   Issue                         str  
 5   Timely response?              str  
 6   Category                      str  
 7   cleaned_narrative             str  
 8   token_count                   int64
 9   Urgency                       str  
dtypes: int64(2), str(8)
memory usage: 6.4 GB


In [8]:
df['Urgency'].value_counts()

Urgency
Low         1208768
Medium      1021745
Critical     807656
High         603464
Name: count, dtype: int64

In [9]:
df.to_csv('cleaned_complaints_with_labels.csv', index=False)

Does the trained urgency model outperform the rule-based keyword function on held-out complaints?
You can test this by comparing model predictions vs keyword function predictions on the same complaints.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


x = df['Consumer complaint narrative']
y = df['Urgency']

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=100000,
    min_df=5,
    sublinear_tf=True
)

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)



In [11]:
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    random_state=42
)

lr.fit(x_train_vec, y_train)

y_pred = lr.predict(x_test_vec)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Critical       0.96      0.95      0.95    161531
        High       0.92      0.94      0.93    120693
         Low       0.99      0.99      0.99    241754
      Medium       0.97      0.96      0.96    204349

    accuracy                           0.96    728327
   macro avg       0.96      0.96      0.96    728327
weighted avg       0.96      0.96      0.96    728327



In [16]:
samples = [
    "I have been trying to reach someone at the bank for three weeks regarding an error on my statement. Every time I call I am placed on hold for over an hour and then disconnected. The error is for $45 and I would like it corrected.",
    "Someone has been using my social security number to open credit accounts. I have already filed a police report and contacted all three credit bureaus. New fraudulent accounts keep appearing every month despite my fraud alerts.",
    "I would like to update my phone number on file with your institution. My previous number is no longer active and I want to ensure I receive account notifications correctly.",
    "My mortgage servicer applied my payment to the wrong account for three consecutive months. My credit score has dropped significantly and I am now being reported as delinquent despite making all payments on time.",
    "I cannot access my account and all my direct deposit funds are inside. My rent is due tomorrow and I have no other way to pay. The bank says they need 5 business days to investigate a flag on my account.",
    "I received a collections notice for a medical bill I paid two years ago. I have the receipt showing full payment. The collections agency is reporting this to the credit bureaus incorrectly."
]

cleaned_samples = [clean_preprocessed_text(s) for s in samples]

In [18]:
for i, (raw, cleaned) in enumerate(zip(samples, cleaned_samples)):
    keyword_pred = urgency_level(raw)
    model_pred = lr.predict(vectorizer.transform([cleaned]))[0]
    print(f"Sample {i+1} | Keyword: {keyword_pred} | Model: {model_pred}")

Sample 1 | Keyword: Low | Model: Low
Sample 2 | Keyword: Critical | Model: Critical
Sample 3 | Keyword: Low | Model: Low
Sample 4 | Keyword: High | Model: Low
Sample 5 | Keyword: Critical | Model: Critical
Sample 6 | Keyword: Medium | Model: Medium


"Urgency model achieves 0.95 F1 on keyword-labeled test data but shows evidence of label leakage — model predictions closely mirror keyword function outputs rather than generalizing to novel urgency signals. This is a known limitation of weak supervision approaches."

In [19]:
import joblib

joblib.dump(lr, 'urgency_lr_model.pkl')
joblib.dump(vectorizer, 'urgency_tfidf_vectorizer.pkl')

['urgency_tfidf_vectorizer.pkl']

In [1]:
import pandas as pd

df = pd.read_csv('cleaned_complaints_with_labels.csv', encoding='utf-8', engine='python')

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_full = df['cleaned_narrative']
y_full = df['Category']

x_train_full, x_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

vectorizer_full = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=100000,
    min_df=5,
    sublinear_tf=True
)

x_train_vec_full = vectorizer_full.fit_transform(x_train_full)
x_test_vec_full = vectorizer_full.transform(x_test_full)

lr_full = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=42
)

lr_full.fit(x_train_vec_full, y_train_full)

c:\Users\Drashti Shah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\Drashti Shah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [3]:
from sklearn.metrics import classification_report

y_pred_full = lr_full.predict(x_test_vec_full)

print(classification_report(y_test_full, y_pred_full))

                           precision    recall  f1-score   support

       Banking & Accounts       0.71      0.86      0.78     37781
     Cards & Transactions       0.73      0.78      0.76     46811
         Credit Reporting       0.98      0.88      0.93    478583
          Debt Collection       0.69      0.86      0.76     82936
                    Loans       0.74      0.90      0.81     59077
Money Transfer & Payments       0.77      0.84      0.80     23139

                 accuracy                           0.87    728327
                macro avg       0.77      0.85      0.81    728327
             weighted avg       0.89      0.87      0.88    728327



In [4]:
import joblib

joblib.dump(lr_full, 'lr_category_full.pkl')
joblib.dump(vectorizer_full, 'tfidf_category_full.pkl')

['tfidf_category_full.pkl']

In [5]:
routing_map = {
    "Credit Reporting": {
        "Critical": "Fraud & Identity Protection",
        "High": "Fraud & Identity Protection",
        "Medium": "Credit Reporting Support",
        "Low": "Credit Reporting Support"
    },
    "Loans": {
        "Critical": "Emergency Loan Services",
        "High": "Loan Resolution Team",
        "Medium": "Standard Loan Servicing",
        "Low": "Standard Loan Servicing"
    },
    "Cards & Transactions": {
        "Critical": "Card Fraud Team",
        "High": "Card Disputes Team",
        "Medium": "Card Services Support",
        "Low": "Card Services Support"
    },
    "Banking & Accounts": {
        "Critical": "Account Security Team",
        "High": "Account Resolution Team",
        "Medium": "Account Services Support",
        "Low": "Account Services Support"
    },
    "Debt Collection": {
        "Critical": "Legal & Collections Team",
        "High": "Legal & Collections Team",
        "Medium": "Collections Support",
        "Low": "Collections Support"
    },
    "Money Transfer & Payments": {
        "Critical": "Payment Fraud Team",
        "High": "Payment Investigation Team",
        "Medium": "Payment Services Support",
        "Low": "Payment Services Support"
    }
}

In [6]:
def get_department(category,urgency):

    return routing_map[category][urgency]

In [21]:
def analyze_complaint(text):

    text = text.lower()

    urgency = urgency_level(text)

    cleaned_text = clean_preprocessed_text(text)

    predicted_category = lr_full.predict(vectorizer_full.transform([cleaned_text]))[0]

    department = get_department(predicted_category, urgency)

    model_confidence = lr_full.predict_proba(vectorizer_full.transform([cleaned_text]))[0].max()

    return {
        'category': predicted_category,
        'urgency': urgency,
        'department': department,
        'confidence': model_confidence
    }

In [22]:
text = "I sent $800 through Zelle to pay my landlord last week. The money left my account but my landlord never received it. I have contacted my bank three times and they keep telling me to wait 3-5 business days. It has now been 10 days and the money has disappeared."

result = analyze_complaint(text)

print(result)

{'category': 'Money Transfer & Payments', 'urgency': 'High', 'department': 'Payment Investigation Team', 'confidence': np.float64(0.9945785914370718)}
